<a href="https://colab.research.google.com/github/paplpap/zero-to-AI/blob/main/8%EC%9E%A5_zero%EB%B6%80%ED%84%B0_ai%2C_%EB%A9%80%ED%8B%B0%ED%84%B4_%EC%98%88%EC%8B%9C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install -qU langchain-openai langchain-community chromadb langchain-core langchainhub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.7/88.7 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 36.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.1/515.1 kB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 63.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 39.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6

In [3]:
!pip install -qU langchainhub

In [4]:
!pip install -U ddgs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 kB 890.0 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 13.6 MB/s eta 0:00:00


In [5]:
import os
import warnings
from datetime import UTC
import datetime

# 1. 경고 메시지 끄기
warnings.filterwarnings("ignore", category=DeprecationWarning)

from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import Chroma
from langchain_core.tools import create_retriever_tool
from langchain_core.tools import Tool
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.documents import Document
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_classic.agents import AgentExecutor
from langchain_classic.agents import create_tool_calling_agent


In [6]:
# 1. API 키 설정 (보안상 환경변수 권장)
os.environ["OPENAI_API_KEY"] = "OPENAI_API_KEY"

# [Tool 2] 내부 함수 (내 로컬의 계산기)
def calculate_special_bonus(base_salary: str):
    """급여에 따른 특별 보너스를 계산합니다."""
    salary = int(base_salary)
    return f"당신의 특별 보너스는 {salary * 0.15}원입니다!"

bonus_tool = Tool(
    name="bonus_calculator",
    func=calculate_special_bonus,
    description="사용자의 급여를 입력하면 보너스를 계산해줍니다. 숫자만 입력하세요."
)

# [Tool 3] 외부 API (실시간 웹 검색)
search = DuckDuckGoSearchRun()
search_tool = Tool(
    name="web_search",
    func=search.run,
    description="오늘의 날씨, 뉴스 등 실시간 정보가 필요할 때 사용하세요."
)

# 도구 바구니에 모두 담기
tools = [ bonus_tool, search_tool]

In [7]:
# 에이전트 설정
llm = ChatOpenAI(model="gpt-4o", temperature=0)

# 직접 클라이언트를 생성해서 pull 합니다.
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant"),
    MessagesPlaceholder(variable_name="chat_history", optional=True),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

agent = create_tool_calling_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)


In [8]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

# 1. 메모리 저장소 생성 (세션별로 대화 내역이 저장됩니다)
store = {}

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

# 2. 기존 에이전트 실행기에 '기억' 입히기
# input_messages_key: 사용자의 질문이 들어가는 키
# history_messages_key: 프롬프트 내 MessagesPlaceholder와 매칭될 키
agent_with_chat_history = RunnableWithMessageHistory(
    agent_executor,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history",
)

# 3. 세션 ID를 이용한 연속 대화 테스트
config = {"configurable": {"session_id": "user_123"}}

# 첫 번째 질문
agent_with_chat_history.invoke({"input": "내 보너스가 100만 원이었나?"}, config=config)

# 두 번째 질문 (기억을 바탕으로 답변!)
agent_with_chat_history.invoke({"input": "방금 말한 보너스에서 세금 10% 떼면 얼마지?"}, config=config)




> Entering new AgentExecutor chain...
보너스 금액을 확인하려면 급여 정보를 입력해 주시면 보너스를 계산해 드릴 수 있습니다. 급여를 숫자로 입력해 주세요.

> Finished chain.


> Entering new AgentExecutor chain...
보너스 금액이 100만 원이라고 가정하면, 세금 10%를 떼면 다음과 같이 계산할 수 있습니다.

1. 세금 금액: 100만 원 * 10% = 10만 원
2. 세금 공제 후 보너스: 100만 원 - 10만 원 = 90만 원

따라서 세금 10%를 공제한 후의 보너스 금액은 90만 원입니다.

> Finished chain.


{'input': '방금 말한 보너스에서 세금 10% 떼면 얼마지?',
 'chat_history': [HumanMessage(content='내 보너스가 100만 원이었나?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='보너스 금액을 확인하려면 급여 정보를 입력해 주시면 보너스를 계산해 드릴 수 있습니다. 급여를 숫자로 입력해 주세요.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])],
 'output': '보너스 금액이 100만 원이라고 가정하면, 세금 10%를 떼면 다음과 같이 계산할 수 있습니다.\n\n1. 세금 금액: 100만 원 * 10% = 10만 원\n2. 세금 공제 후 보너스: 100만 원 - 10만 원 = 90만 원\n\n따라서 세금 10%를 공제한 후의 보너스 금액은 90만 원입니다.'}

In [9]:
agent_with_chat_history.invoke({"input": "그러면 내가 질문한 원금과 세금을 제외한 금액을 더해줘"}, config=config)




> Entering new AgentExecutor chain...
당신이 질문한 원금은 100만 원이고, 세금을 제외한 금액은 90만 원입니다. 이 두 금액을 더하면:

100만 원 + 90만 원 = 190만 원

따라서 원금과 세금을 제외한 금액을 더한 총액은 190만 원입니다.

> Finished chain.


{'input': '그러면 내가 질문한 원금과 세금을 제외한 금액을 더해줘',
 'chat_history': [HumanMessage(content='내 보너스가 100만 원이었나?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='보너스 금액을 확인하려면 급여 정보를 입력해 주시면 보너스를 계산해 드릴 수 있습니다. 급여를 숫자로 입력해 주세요.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content='방금 말한 보너스에서 세금 10% 떼면 얼마지?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='보너스 금액이 100만 원이라고 가정하면, 세금 10%를 떼면 다음과 같이 계산할 수 있습니다.\n\n1. 세금 금액: 100만 원 * 10% = 10만 원\n2. 세금 공제 후 보너스: 100만 원 - 10만 원 = 90만 원\n\n따라서 세금 10%를 공제한 후의 보너스 금액은 90만 원입니다.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])],
 'output': '당신이 질문한 원금은 100만 원이고, 세금을 제외한 금액은 90만 원입니다. 이 두 금액을 더하면:\n\n100만 원 + 90만 원 = 190만 원\n\n따라서 원금과 세금을 제외한 금액을 더한 총액은 190만 원입니다.'}

In [10]:
agent_with_chat_history.invoke({"input": "그러면 세션은 정확히 어디에 저장되는거지?"}, config=config)



> Entering new AgentExecutor chain...
세션 데이터는 일반적으로 서버 측에 저장되며, 이는 웹 애플리케이션의 구조와 구현 방식에 따라 다를 수 있습니다. 일반적인 세션 저장 방식은 다음과 같습니다:

1. **서버 메모리**: 세션 데이터를 서버의 메모리에 저장합니다. 이는 빠르지만 서버가 재시작되면 데이터가 사라질 수 있습니다.

2. **데이터베이스**: 세션 데이터를 데이터베이스에 저장합니다. 이는 영구적이며 여러 서버 간에 세션을 공유할 수 있습니다.

3. **파일 시스템**: 서버의 파일 시스템에 세션 데이터를 저장합니다. 이는 간단하지만 파일 입출력 속도가 느릴 수 있습니다.

4. **캐시 시스템**: Redis나 Memcached와 같은 인메모리 데이터 저장소를 사용하여 세션 데이터를 저장합니다. 이는 빠르고 확장성이 좋습니다.

5. **쿠키**: 세션 데이터를 클라이언트 측 쿠키에 저장할 수도 있지만, 보안상의 이유로 민감한 정보를 저장하는 것은 피해야 합니다.

세션 저장 방식은 애플리케이션의 요구 사항, 성능, 보안 고려 사항에 따라 선택됩니다.

> Finished chain.


{'input': '그러면 세션은 정확히 어디에 저장되는거지?',
 'chat_history': [HumanMessage(content='내 보너스가 100만 원이었나?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='보너스 금액을 확인하려면 급여 정보를 입력해 주시면 보너스를 계산해 드릴 수 있습니다. 급여를 숫자로 입력해 주세요.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content='방금 말한 보너스에서 세금 10% 떼면 얼마지?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='보너스 금액이 100만 원이라고 가정하면, 세금 10%를 떼면 다음과 같이 계산할 수 있습니다.\n\n1. 세금 금액: 100만 원 * 10% = 10만 원\n2. 세금 공제 후 보너스: 100만 원 - 10만 원 = 90만 원\n\n따라서 세금 10%를 공제한 후의 보너스 금액은 90만 원입니다.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content='그러면 내가 질문한 원금과 세금을 제외한 금액을 더해줘', additional_kwargs={}, response_metadata={}),
  AIMessage(content='당신이 질문한 원금은 100만 원이고, 세금을 제외한 금액은 90만 원입니다. 이 두 금액을 더하면:\n\n100만 원 + 90만 원 = 190만 원\n\n따라서 원금과 세금을 제외한 금액을 더한 총액은 190만 원입니다.', additional_kwargs={}, response_metadata={}, tool_calls=[], inval

In [11]:
agent_with_chat_history.invoke({"input": "그러면 store를 DB에 넣어놓으면 사용자 관리가 좋겠군"}, config=config)



> Entering new AgentExecutor chain...
네, 세션 데이터를 데이터베이스에 저장하는 것은 여러 가지 장점이 있습니다:

1. **영속성**: 서버가 재시작되거나 장애가 발생하더라도 세션 데이터가 유지됩니다.

2. **확장성**: 여러 대의 서버가 동일한 데이터베이스를 참조할 수 있으므로, 로드 밸런싱을 통해 트래픽을 분산시킬 수 있습니다.

3. **중앙 집중 관리**: 모든 세션 데이터가 중앙의 데이터베이스에 저장되므로 관리가 용이합니다.

4. **보안**: 데이터베이스의 보안 기능을 활용하여 세션 데이터를 보호할 수 있습니다.

5. **데이터 분석**: 세션 데이터를 분석하여 사용자 행동을 이해하고 개선할 수 있습니다.

하지만, 데이터베이스에 세션을 저장할 때는 성능에 주의해야 합니다. 데이터베이스에 대한 빈번한 읽기/쓰기가 발생할 수 있으므로, 적절한 인덱싱과 캐싱 전략을 사용하는 것이 중요합니다. Redis와 같은 인메모리 데이터베이스를 사용하면 성능을 향상시킬 수 있습니다.

> Finished chain.


{'input': '그러면 store를 DB에 넣어놓으면 사용자 관리가 좋겠군',
 'chat_history': [HumanMessage(content='내 보너스가 100만 원이었나?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='보너스 금액을 확인하려면 급여 정보를 입력해 주시면 보너스를 계산해 드릴 수 있습니다. 급여를 숫자로 입력해 주세요.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content='방금 말한 보너스에서 세금 10% 떼면 얼마지?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='보너스 금액이 100만 원이라고 가정하면, 세금 10%를 떼면 다음과 같이 계산할 수 있습니다.\n\n1. 세금 금액: 100만 원 * 10% = 10만 원\n2. 세금 공제 후 보너스: 100만 원 - 10만 원 = 90만 원\n\n따라서 세금 10%를 공제한 후의 보너스 금액은 90만 원입니다.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content='그러면 내가 질문한 원금과 세금을 제외한 금액을 더해줘', additional_kwargs={}, response_metadata={}),
  AIMessage(content='당신이 질문한 원금은 100만 원이고, 세금을 제외한 금액은 90만 원입니다. 이 두 금액을 더하면:\n\n100만 원 + 90만 원 = 190만 원\n\n따라서 원금과 세금을 제외한 금액을 더한 총액은 190만 원입니다.', additional_kwargs={}, response_metadata={}, tool_calls=

In [12]:
agent_with_chat_history.invoke({"input": "유저를 바꾸면 응답또한 바뀌는걸 볼 수 있습니다."}, config={"configurable": {"session_id": "user_456"}})



> Entering new AgentExecutor chain...
네, 맞습니다. 사용자가 제공하는 정보나 요청에 따라 응답이 달라질 수 있습니다. 사용자의 질문이나 요청에 따라 적절한 정보를 제공하기 위해 최선을 다하고 있습니다. 특정한 질문이나 요청이 있으시면 말씀해 주세요!

> Finished chain.


{'input': '유저를 바꾸면 응답또한 바뀌는걸 볼 수 있습니다.',
 'chat_history': [],
 'output': '네, 맞습니다. 사용자가 제공하는 정보나 요청에 따라 응답이 달라질 수 있습니다. 사용자의 질문이나 요청에 따라 적절한 정보를 제공하기 위해 최선을 다하고 있습니다. 특정한 질문이나 요청이 있으시면 말씀해 주세요!'}

In [13]:
config

{'configurable': {'session_id': 'user_123'}}

In [14]:
store

{'user_123': InMemoryChatMessageHistory(messages=[HumanMessage(content='내 보너스가 100만 원이었나?', additional_kwargs={}, response_metadata={}), AIMessage(content='보너스 금액을 확인하려면 급여 정보를 입력해 주시면 보너스를 계산해 드릴 수 있습니다. 급여를 숫자로 입력해 주세요.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='방금 말한 보너스에서 세금 10% 떼면 얼마지?', additional_kwargs={}, response_metadata={}), AIMessage(content='보너스 금액이 100만 원이라고 가정하면, 세금 10%를 떼면 다음과 같이 계산할 수 있습니다.\n\n1. 세금 금액: 100만 원 * 10% = 10만 원\n2. 세금 공제 후 보너스: 100만 원 - 10만 원 = 90만 원\n\n따라서 세금 10%를 공제한 후의 보너스 금액은 90만 원입니다.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='그러면 내가 질문한 원금과 세금을 제외한 금액을 더해줘', additional_kwargs={}, response_metadata={}), AIMessage(content='당신이 질문한 원금은 100만 원이고, 세금을 제외한 금액은 90만 원입니다. 이 두 금액을 더하면:\n\n100만 원 + 90만 원 = 190만 원\n\n따라서 원금과 세금을 제외한 금액을 더한 총액은 190만 원입니다.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[